# Interactive Stock Valuation & Risk Analyzer  
**Core Calculations**  
AMA 106 – Introduction to Financial Economics  
Personal Project by CJ  
March 2026

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np

# ====================== INPUTS FROM CLASS REPORT ======================
ticker = "NVDA"
shares_outstanding = 24.3          # billion shares

# DCF (matching your class numbers)
base_fcf_2025 = 60.85
growth_rates = [0.30, 0.30, 0.25, 0.20, 0.15]   # 2026 to 2030
rf = 0.0424
mrp = 0.06
beta = 2.31

# States of Nature (exactly from your report)
p_boom = 0.30
p_normal = 0.50
p_recession = 0.20
r_boom = 0.60
r_normal = 0.20
r_recession = -0.40

A = 4.0   # Risk aversion coefficient

print("✅ Notebook ready for AMA 106 core calculations")

✅ Notebook ready for AMA 106 core calculations


## Make sure when you run the code, your connected to the internet since the output gives live price of nvdia 

In [4]:
# ====================== LIVE MARKET DATA (Chapter 2) ======================
# More reliable way to download data
ticker_obj = yf.Ticker(ticker)
data = ticker_obj.history(period="2y")

if data.empty:
    print("⚠️  yfinance could not download live data. Using fallback price from your class report.")
    current_price_live = 180.00
else:
    current_price_live = data['Close'].iloc[-1]   # 'Close' is more stable than 'Adj Close'

print(f"Live Current Price of {ticker}: ${current_price_live:.2f}\n")

# ====================== DCF VALUATION (Chapter 2 - Time Value of Money) ======================
fcf = [base_fcf_2025]
for g in growth_rates:
    fcf.append(fcf[-1] * (1 + g))
fcf = fcf[1:]

discount_factors = [(1 + rf) ** (i+1) for i in range(5)]
pv_fcf = [fcf[i] / discount_factors[i] for i in range(5)]

tv = fcf[-1] * (1 + 0.03) / (rf + mrp*beta - 0.03)
pv_tv = tv / discount_factors[-1]

total_pv = sum(pv_fcf) + pv_tv
intrinsic_per_share = total_pv / shares_outstanding

print("=== DCF VALUATION (Chapter 2 - Time Value of Money) ===")
print(f"Intrinsic Value per share : ${intrinsic_per_share:.2f}")
print(f"Current Market Price      : ${current_price_live:.2f}")
print(f"Valuation                 : {'Undervalued' if intrinsic_per_share > current_price_live else 'Overvalued'} "
      f"by {abs(intrinsic_per_share - current_price_live)/current_price_live*100:.1f}%\n")

# ====================== STATES OF NATURE & PAYOFF VECTOR (Chapter 3) ======================
print("=== STATES OF NATURE & PAYOFF VECTOR (Chapter 3) ===")
payoffs = [100000 * (1 + r) for r in [r_boom, r_normal, r_recession]]
df_states = pd.DataFrame({
    "State": ["Boom", "Normal", "Recession"],
    "Probability": [p_boom, p_normal, p_recession],
    "Return": [r_boom, r_normal, r_recession],
    "Payoff (PHP)": payoffs
})
display(df_states.style.format({"Probability": "{:.1%}", "Return": "{:.1%}", "Payoff (PHP)": "₱{:,.0f}"}))

# ====================== EXPECTED RETURN & RISK (Chapter 4) ======================
e_r = p_boom*r_boom + p_normal*r_normal + p_recession*r_recession
var = (p_boom*(r_boom - e_r)**2 + 
       p_normal*(r_normal - e_r)**2 + 
       p_recession*(r_recession - e_r)**2)
std = np.sqrt(var)
risk_premium = e_r - rf

print(f"\nExpected Return     : {e_r:.1%}")
print(f"Standard Deviation  : {std:.1%}")
print(f"Risk Premium        : {risk_premium:.1%}")

# ====================== EXPECTED UTILITY (Chapter 3) ======================
u_neutral = e_r
u_averse  = e_r - 0.5 * A * var

print(f"\nRisk-Neutral Utility : {u_neutral:.1%}")
print(f"Risk-Averse Utility  : {u_averse:.1%}")
print(f"Risk-Free Rate       : {rf:.1%}")

Live Current Price of NVDA: $175.20

=== DCF VALUATION (Chapter 2 - Time Value of Money) ===
Intrinsic Value per share : $63.46
Current Market Price      : $175.20
Valuation                 : Overvalued by 63.8%

=== STATES OF NATURE & PAYOFF VECTOR (Chapter 3) ===


,State,Probability,Return,Payoff (PHP)
0,Boom,30.0%,60.0%,"₱160,000"
1,Normal,50.0%,20.0%,"₱120,000"
2,Recession,20.0%,-40.0%,"₱60,000"



Expected Return     : 20.0%
Standard Deviation  : 34.6%
Risk Premium        : 15.8%

Risk-Neutral Utility : 20.0%
Risk-Averse Utility  : -4.0%
Risk-Free Rate       : 4.2%
